In [1]:
#the preprocessing environment works for this script
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import pandas as pd

# Replace this with the path to one of your .h5 files
path_to_Reference = "/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/Inference/2.demand_allocation/gridalloc/results/0197_N2780244E4431460_80689_2_2_PV100_HP100_EV100_VarTar0_CapPr0_pwrflw.h5"

# See what's inside the HDF5 file
with pd.HDFStore(path_to_Reference, mode='r') as store:
    keys = store.keys()
    print("Available tables in this file:")
    for k in keys:
        print(f" - {k}")


Available tables in this file:
 - /performance
 - /urbs_out/tsam/kept_timesteps
 - /urbs_out/reduced_data/buy_sell_price
 - /urbs_out/reduced_data/commodity
 - /urbs_out/reduced_data/demand
 - /urbs_out/reduced_data/eff_factor
 - /urbs_out/reduced_data/global_prop
 - /urbs_out/reduced_data/process
 - /urbs_out/reduced_data/process_commodity
 - /urbs_out/reduced_data/site
 - /urbs_out/reduced_data/storage
 - /urbs_out/reduced_data/supim
 - /urbs_out/reduced_data/type_period
 - /urbs_out/reduced_data/weather
 - /urbs_out/MILP/cap_pro
 - /urbs_out/MILP/cap_pro_new
 - /urbs_out/MILP/cap_sto_c
 - /urbs_out/MILP/cap_sto_c_new
 - /urbs_out/MILP/cap_sto_p
 - /urbs_out/MILP/cap_sto_p_new
 - /urbs_out/MILP/costs
 - /urbs_out/MILP/dt
 - /urbs_out/MILP/e_co_buy
 - /urbs_out/MILP/e_co_sell
 - /urbs_out/MILP/e_pro_in
 - /urbs_out/MILP/e_pro_out
 - /urbs_out/MILP/e_sto_con
 - /urbs_out/MILP/e_sto_in
 - /urbs_out/MILP/e_sto_out
 - /urbs_out/MILP/pro_cap_expands
 - /urbs_out/MILP/tau_pro
 - /urbs_out/M

In [4]:
#the preprocessing environment works for this script
import os
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

import pandas as pd

# Replace this with the path to one of your .h5 files
path_2_NEW = "/home/pedro/Linux-AntigravityProjects/SurroGrid-linux/Inference/2.demand_allocation/gridalloc/results/0197_N2780244E4431460_80689_2_2_DmdAllocated.h5"

# See what's inside the HDF5 file
with pd.HDFStore(path_2_NEW, mode='r') as store:
    keys = store.keys()
    print("Available tables in this file:")
    for k in keys:
        print(f" - {k}")


Available tables in this file:
 - /performance
 - /urbs_in/demand
 - /urbs_in/eff_factor
 - /urbs_in/process
 - /urbs_in/supim
 - /urbs_in/weather
 - /raw_data/buildings
 - /raw_data/consumers
 - /raw_data/region
 - /raw_data/weather
 - /pwrflw/output/pre/demand_import
 - /pwrflw/output/pre/line_loads
 - /pwrflw/output/pre/vm
 - /pwrflw/input/demand_pre


In [ ]:
# Helper function for comparative plotting (full-year daily mean + 1-week hourly zoom)
import numpy as np
import matplotlib.pyplot as plt

def plot_feature_comparison(s_ref, s_new, title, ylabel, zoom_start=240, zoom_hours=168, is_power_kw=False):
    """Plot daily-mean full year and 1-week hourly zoom comparing Reference vs New."""
    s_ref = pd.Series(s_ref).reset_index(drop=True)
    s_new = pd.Series(s_new).reset_index(drop=True)
    corr = np.corrcoef(s_ref.values, s_new.values)[0, 1]

    # Summary metrics
    print(f"{title}")
    print(f"  Reference: min={s_ref.min():.4f}, mean={s_ref.mean():.4f}, max={s_ref.max():.4f}, NaNs={s_ref.isna().sum()}")
    print(f"  New:       min={s_new.min():.4f}, mean={s_new.mean():.4f}, max={s_new.max():.4f}, NaNs={s_new.isna().sum()}")
    if is_power_kw:
        print(f"  Annual sum: Reference={s_ref.sum()/1000:.2f} MWh | New={s_new.sum()/1000:.2f} MWh ({(s_new.sum()/s_ref.sum()-1)*100:+.2f}%)")
    print(f"  Pearson r: {corr:.4f}")

    # Daily means for full-year overview
    days = np.arange(len(s_ref)) // 24
    daily_ref = s_ref.groupby(days).mean()
    daily_new = s_new.groupby(days).mean()

    fig, axes = plt.subplots(1, 2, figsize=(15, 4))

    # Left: Full year (daily mean)
    axes[0].plot(daily_ref.index, daily_ref.values, label="Reference (GT)", alpha=0.85, linewidth=1.2)
    axes[0].plot(daily_new.index, daily_new.values, label="New (Inference)", alpha=0.80, linewidth=1.2, linestyle="--")
    axes[0].set_title(f"{title} — Full Year (Daily Mean)")
    axes[0].set_xlabel("Day of Year")
    axes[0].set_ylabel(ylabel)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    # Right: 1-week hourly zoom
    zoom_end = zoom_start + zoom_hours
    hrs = np.arange(zoom_start, zoom_end)
    axes[1].plot(hrs, s_ref.iloc[zoom_start:zoom_end].values, label="Reference (GT)", alpha=0.85, linewidth=1.3)
    axes[1].plot(hrs, s_new.iloc[zoom_start:zoom_end].values, label="New (Inference)", alpha=0.80, linewidth=1.3, linestyle="--")
    axes[1].set_title(f"{title} — 1-Week Zoom (Hours {zoom_start}–{zoom_end})")
    axes[1].set_xlabel("Hour of Year")
    axes[1].set_ylabel(ylabel)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Display the Electrical load time series on the transformer before urbs for both the 
# reference and the new .h5 files
elec_ref = pd.read_hdf(path_to_Reference, key="/pwrflw/output/pre/demand_import")["p_mw"]
elec_new = pd.read_hdf(path_2_NEW, key="/pwrflw/output/pre/demand_import")["p_mw"]

plot_feature_comparison(
    elec_ref, elec_new,
    title="Pre-URBS Transformer Active Load (demand_net_active_pre)",
    ylabel="Active Power [MW]",
    zoom_start=240,  # mid-January week
)


In [ ]:
# Display the aggregated DHW Time series for both the 
# reference and the new .h5 files
dmd_ref = pd.read_hdf(path_to_Reference, key="/urbs_in/demand")
dmd_new = pd.read_hdf(path_2_NEW, key="/urbs_in/demand")

dhw_ref = dmd_ref.loc[:, dmd_ref.columns.get_level_values(1).str.startswith("water_heat")].sum(axis=1)
dhw_new = dmd_new.loc[:, dmd_new.columns.get_level_values(1).str.startswith("water_heat")].sum(axis=1)

plot_feature_comparison(
    dhw_ref, dhw_new,
    title="Aggregated DHW Demand (heat_water)",
    ylabel="Thermal Power [kW]",
    zoom_start=240,
    is_power_kw=True,
)


In [ ]:
# Display the aggregated Space Heat Time series for both the 
# reference and the new .h5 files
sh_ref = dmd_ref.loc[:, dmd_ref.columns.get_level_values(1).str.startswith("space_heat")].sum(axis=1)
sh_new = dmd_new.loc[:, dmd_new.columns.get_level_values(1).str.startswith("space_heat")].sum(axis=1)

plot_feature_comparison(
    sh_ref, sh_new,
    title="Aggregated Space Heat Demand (heat_space)",
    ylabel="Thermal Power [kW]",
    zoom_start=240,
    is_power_kw=True,
)


In [ ]:
# Display the Grid's COP (average for every building, weighted by the building's total heat demand)
# Time serie for both the reference and the new .h5 files
def compute_weighted_cop(h5_path):
    """Exact get_heat_and_cop logic from GridForecast/0_preprocessing/0.extract_ML_data.ipynb."""
    with pd.HDFStore(h5_path, mode="r") as store:
        demand = store["urbs_in/demand"]
        eff_factor = store["urbs_in/eff_factor"]

    space_heat = demand.loc[:, demand.columns.get_level_values(1).str.startswith("space_heat")].droplevel(1, axis=1)
    water_heat = demand.loc[:, demand.columns.get_level_values(1).str.startswith("water_heat")].droplevel(1, axis=1)
    hp_cop = eff_factor.loc[:, eff_factor.columns.get_level_values(1).str.startswith("heatpump_air")].droplevel(1, axis=1)

    total_heat_cols = space_heat + water_heat
    denom = total_heat_cols.sum(axis=1)
    weighted_sum = hp_cop.multiply(total_heat_cols).sum(axis=1)

    weighted_cop = pd.Series(np.nan, index=denom.index, dtype=float)
    nonzero = denom != 0
    weighted_cop.loc[nonzero] = weighted_sum.loc[nonzero] / denom.loc[nonzero]
    weighted_cop.loc[~nonzero] = hp_cop.mean(axis=1).loc[~nonzero]
    return weighted_cop

cop_ref = compute_weighted_cop(path_to_Reference)
cop_new = compute_weighted_cop(path_2_NEW)

plot_feature_comparison(
    cop_ref, cop_new,
    title="Grid Weighted Average Heat Pump COP (cop_avg)",
    ylabel="COP [-]",
    zoom_start=240,
)


In [ ]:
# Display the Expected PV Production (PV_prod_expected) time series for both files
def compute_pv_prod_expected(h5_path, fraction_of_total=0.6766):
    """Exact create_PV_prod_up_to logic from GridForecast/0_preprocessing/0.extract_ML_data.ipynb."""
    with pd.HDFStore(h5_path, mode="r") as store:
        supim = store["urbs_in/supim"]
        processes = store["urbs_in/process"]

    pv_100 = processes[processes["Process"].str.startswith("Rooftop PV")].set_index(["Site", "Process"])["cap-up"].to_frame().T
    cap_map = {
        (site, cap_proc.replace("Rooftop PV", "solar")): pv_100[(site, cap_proc)].iloc[0]
        for site, cap_proc in pv_100.columns
    }
    cap_series = pd.Series([cap_map.get((site, proc), pd.NA) for site, proc in supim.columns], index=supim.columns)
    df_scaled = supim * cap_series

    target_energy = df_scaled.sum().sum() * fraction_of_total
    order = supim.sum(axis=0).sort_values(ascending=False).index
    energy_sums = df_scaled.sum(axis=0)

    selected_ts = []
    total_energy = 0.0
    for col in order:
        col_energy = energy_sums[col]
        if total_energy + col_energy <= target_energy:
            selected_ts.append(df_scaled[col])
            total_energy += col_energy
        else:
            frac = (target_energy - total_energy) / col_energy
            selected_ts.append(df_scaled[col] * frac)
            break

    return pd.concat(selected_ts, axis=1).sum(axis=1)

pv_ref = compute_pv_prod_expected(path_to_Reference)
pv_new = compute_pv_prod_expected(path_2_NEW)

plot_feature_comparison(
    pv_ref, pv_new,
    title="Expected PV Production (PV_prod_expected)",
    ylabel="Power [kW]",
    zoom_start=4320,  # late-June summer week
    is_power_kw=True,
)


In [ ]:
# Check if there are any remaining active MV-level buildings in the .h5 files
# Thresholds (consistent with disconnect_MV_buildings.py):
#   - Pure non-residential (residential_floor_area == 0): peak electricity > 100 kW
#   - Pure residential / mixed (residential_floor_area > 0): peak electricity > 250 kW
import io
import json
import h5py

def check_mv_buildings(h5_path, label, nonres_thresh_kw=100.0, res_thresh_kw=250.0):
    df_buildings = pd.read_hdf(h5_path, key="/raw_data/buildings")
    df_demand = pd.read_hdf(h5_path, key="/urbs_in/demand")
    df_elec = df_demand.loc[:, df_demand.columns.get_level_values(1) == "electricity"]
    peak_by_bus = {int(float(col[0])): float(df_elec[col].max()) for col in df_elec.columns}

    # Read load table from raw_data/net JSON to inspect in_service status
    with h5py.File(h5_path, "r") as f:
        net_json = f["raw_data/net"][()]
        if isinstance(net_json, bytes):
            net_json = net_json.decode("utf-8")
        net_data = json.loads(net_json)
    load_info = net_data["_object"]["load"]
    df_load = pd.read_json(io.StringIO(load_info["_object"]), orient=load_info["orient"])
    in_service_by_bus = df_load.groupby(df_load["bus"].astype(int))["in_service"].any().to_dict()

    rows = []
    for _, row in df_buildings.iterrows():
        bus = int(float(row["bus"]))
        res_area = 0.0 if pd.isna(row.get("residential_floor_area")) else float(row.get("residential_floor_area"))
        thresh = nonres_thresh_kw if res_area == 0.0 else res_thresh_kw
        peak_kw = peak_by_bus.get(bus, 0.0)
        in_srv = bool(in_service_by_bus.get(bus, True))
        rows.append({
            "bus": bus,
            "use": row.get("use"),
            "res_area_m2": res_area,
            "peak_elec_kw": peak_kw,
            "threshold_kw": thresh,
            "exceeds_threshold": peak_kw > thresh,
            "in_service": in_srv,
        })

    df_check = pd.DataFrame(rows)
    disconnected = df_check[~df_check["in_service"]]
    active_mv_violations = df_check[df_check["exceeds_threshold"] & df_check["in_service"]]

    print(f"=== MV Building Check: {label} ===")
    print(f"  Total buildings:                   {len(df_check)}")
    print(f"  Disconnected (in_service=False):   {len(disconnected)} (buses: {disconnected['bus'].tolist()})")
    print(f"  Active buildings above threshold:  {len(active_mv_violations)}")
    print(f"  Max active building peak (kW):     {df_check.loc[df_check['in_service'], 'peak_elec_kw'].max():.2f} kW")
    if not active_mv_violations.empty:
        print("  WARNING — Remaining active MV buildings found:")
        display(active_mv_violations)
    return df_check.sort_values("peak_elec_kw", ascending=False)

df_mv_ref = check_mv_buildings(path_to_Reference, "Reference (GT)")
print()
df_mv_new = check_mv_buildings(path_2_NEW, "New (Inference)")
print("\nTop 5 highest-peak buildings in New (Inference):")
display(df_mv_new.head(5))
